# 3-08 오늘 급등락 종목과 이유 — 공시 + 뉴스 제목·긍부정

**질문 예시:** "오늘 많이 오른/내린 종목 뭐야? 왜 그런지도 알려줘", "오늘 거래대금 상위 종목은 무슨 뉴스 때문이야?"

그날 **등락률 상위·하위, 거래대금 상위** 종목을 뽑고, 각 종목에
① 전날 장 마감 후부터 당일까지의 **공시**, ② 같은 시간대의 **뉴스 제목과 긍부정**을 붙여 "왜"를 설명합니다.

**왜 DeepSearch로 하나 (CHECK `3_시세스크리너/01` 확장):** CHECK 뉴스에는 종목 태그가 없어 원본은 공시만 붙였습니다.
공시가 없는 급등(테마·풍문)은 설명하지 못했습니다. DeepSearch 뉴스에는 종목 태그(`securities`)와
긍부정 값이 있어 **공시가 없는 종목도 뉴스로 원인**을 확인할 수 있습니다.
또 CHECK 순위 데이터는 최신 거래일만 됐지만, 여기서는 `ASOF`를 바꾸면 **과거 날짜**도 됩니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/3_price_screener/08_movers_with_reasons.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        # 한 페이지가 count 보다 적게 와도(80~90건 흔함) 뒤에 더 있을 수 있다 → 총건수로 판단
        if not docs or page * count >= total:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"   # 기준일(거래일)
TOP = 8               # 그룹별 종목 수
MIN_VALUE = 1e9       # 거래대금 하한(10억) — 거래가 거의 없는 종목의 등락률 제외

## 1. 그날 전 종목 시세와 순위
신규상장 첫날 종목은 등락률 기준이 달라(공모가 대비) 제외합니다.

In [4]:
def stocks(day):
    """일반법인(1)·공기업(2) 보통주 전 종목 하루치. 헬퍼의 all_stocks(common_only=True)는 공기업 8종목
    (한국전력·기업은행·강원랜드 등, company_type_l1='2')을 빼므로 여기서 직접 거릅니다."""
    s = all_stocks(day, common_only=False)
    return s[s["company_type_l1"].isin(["1", "2"]) & s["symbol"].str.endswith("0")].reset_index(drop=True)

snap = stocks(ASOF)
day = snap["date"].max().date()
snap = snap[(snap["date"].dt.date == day) & (snap["volume"] > 0)].copy()
listed = pd.to_datetime(snap["date_listed"].astype(str).str[:10], errors="coerce")
snap = snap[~(listed.dt.date >= day)]   # 상장일이 기준일 이후·당일인 종목 제외 (상장일 모르면 유지)
snap["거래대금(억)"] = snap["value"] / 1e8
print("거래일", day, "/ 종목 수", len(snap))

liquid = snap[snap["value"] >= MIN_VALUE]
groups = {
    "상승 상위": liquid.sort_values("change_pct", ascending=False).head(TOP),
    "하락 상위": liquid.sort_values("change_pct").head(TOP),
    "거래대금 상위": snap.sort_values("value", ascending=False).head(TOP),
}
up, down = (snap["change_pct"] > 0).sum(), (snap["change_pct"] < 0).sum()
print(f"상승 {up} / 하락 {down} / 상한가(≥29.5%) {(snap['change_pct'] >= 29.5).sum()} / 하한가 {(snap['change_pct'] <= -29.5).sum()}")
print(PRICE_NOTICE)

거래일 2026-11-24 / 종목 수 2625
상승 2140 / 하락 426 / 상한가(≥29.5%) 11 / 하한가 0
※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). 거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).


## 2. 공시 — 대상 종목을 묶어 한 번에
전날 15:30 이후 공시는 당일 주가에 반영되므로 함께 봅니다.

In [5]:
prev_day = sorted(index_prices(day - timedelta(days=10), day)["date"].dt.date.unique())[-2]
since = pd.Timestamp(prev_day) + pd.Timedelta(hours=15, minutes=30)
until = pd.Timestamp(day) + pd.Timedelta(hours=23, minutes=59)

targets = pd.concat(groups.values()).drop_duplicates("symbol")
codes = [s.split(":")[-1] for s in targets["symbol"]]
_, disc = doc_search("company", ["disclosure"], "securities.symbol:(" + " or ".join(codes) + ")",
                     prev_day, day, count=100, pages=3)
if not disc.empty:
    disc = disc[(disc["created_at"] >= since) & (disc["created_at"] <= until)]
print(f"공시 {len(disc)}건 ({since:%m-%d %H:%M} ~ {day})")

def disc_titles(code, n=3):
    if disc.empty:
        return ""
    t = disc[disc["symbols"].str.contains(code, na=False)].sort_values("created_at")["title"].drop_duplicates()
    return " / ".join(t.tolist()[-n:])

공시 18건 (11-23 15:30 ~ 2026-11-24)


## 3. 뉴스 — 종목별 제목 상위와 긍부정 비율
종목별로 한 번씩 검색합니다(대상 24개 안팎 → 24회 호출, 태그 검색이 0건이면 제목 검색 1회 추가).
긍부정은 받은 기사(최대 30건, 사진 기사 제외)의 건수입니다.
종목명이 일반 단어와 같은 경우(예: 'DB', 'NEW') 관련 없는 기사가 섞일 수 있어, **제목에 종목명이 들어간 기사**를 우선 보여줍니다.

In [6]:
def news_for(name, code):
    total, df = doc_search("news", [], f"securities.symbol:{code}", prev_day, day, count=30)
    if df.empty:   # 종목 태그가 빠진 기사도 있어, 0건이면 제목 검색으로 한 번 더
        total, df = doc_search("news", [], f'title:"{name}"', prev_day, day, count=30)
    if df.empty:
        return 0, "", ""
    df = df[(df["created_at"] >= since) & (df["created_at"] <= until) & ~df["title"].str.startswith("[포토]")]
    if df.empty:
        return 0, "", ""
    pol = df["polarity"].value_counts()
    mood = f"긍{pol.get('긍정', 0)}·부{pol.get('부정', 0)}·중{pol.get('중립', 0)}"
    in_title = df[df["title"].str.contains(name, regex=False)]
    pick = (in_title if len(in_title) else df).sort_values("created_at", ascending=False)
    return len(df), mood, " / ".join(pick["title"].str[:45].drop_duplicates().head(2))

news = {}
for sym, name in zip(targets["symbol"], targets["name"]):
    news[sym] = news_for(name, sym.split(":")[-1])

## 4. 그룹별 결과

In [7]:
def table(g):
    t = g[["symbol", "name", "market", "change_pct", "거래대금(억)"]].copy()
    t["공시"] = [disc_titles(s.split(":")[-1]) for s in t["symbol"]]
    t["뉴스수"] = [news[s][0] for s in t["symbol"]]
    t["긍부정"] = [news[s][1] for s in t["symbol"]]
    t["대표 뉴스"] = [news[s][2] for s in t["symbol"]]
    t["원인 근거"] = [("공시+뉴스" if a and n else "공시" if a else "뉴스" if n else "없음")
                   for a, n in zip(t["공시"], t["뉴스수"])]
    return t.set_index("name").drop(columns="symbol").round(1)

table(groups["상승 상위"])

,market,change_pct,거래대금(억),공시,뉴스수,긍부정,대표 뉴스,원인 근거
name,,,,,,,,
대성하이텍,KOSDAQ,30.0,56.2,주식등의대량보유상황보고서(일반) / [투자주의]상한가잔량 상위종목,2,긍2·부0·중0,"스마트폰 접던 기술로 '로봇손' 만든다…대성하이텍, 新성장축 확보 / 대성하이텍, 로봇 위탁개발생산 ...",공시+뉴스
모베이스전자,KOSDAQ,30.0,244.0,,0,,,없음
애드바이오텍,KOSDAQ,30.0,20.6,[기재정정]주요사항보고서(타법인주식및출자증권양수결정),5,긍5·부0·중0,"애드바이오텍, 바이오 넘어 K뷰티까지…중앙아시아 시장 공략 나선다 / 애드바이오텍, 4500만 팔로워...",공시+뉴스
KH바텍,KOSDAQ,30.0,325.9,,2,긍1·부0·중1,"'폴더블폰 힌지' KH바텍, 젠슨 황 복귀 훈풍에 '上' [종목+] / [특징주] 삼성 휴머노이드 본...",뉴스
현대오토에버,KOSPI,29.9,849.1,,5,긍5·부0·중0,젠슨황 복귀에 '깐부주' 부활…현대오토에버 상한가[핫종목] / 현대오토에버 25%대 급등…로봇주 동반 강세,뉴스
액트로,KOSDAQ,29.9,33.5,,0,,,없음
휴림로봇,KOSDAQ,29.9,1515.7,[투자주의]투자경고종목 지정예고 / 공매도 과열종목 지정(공매도 거래 금지 적용),3,긍2·부0·중1,엔비디아·삼전發 훈풍에 로봇주 '불기둥'…휴림로봇 이틀째 '上'[핫종목] / [기업이슈] 휴림로봇 상...,공시+뉴스
한국피아이엠,KOSDAQ,29.9,76.7,,0,,,없음


In [8]:
table(groups["하락 상위"])

,market,change_pct,거래대금(억),공시,뉴스수,긍부정,대표 뉴스,원인 근거
name,,,,,,,,
케이아이이,KOSPI,-21.9,13.8,,0,,,없음
아리바이오랩,KOSDAQ,-17.2,250.2,[기재정정]주요사항보고서(유상증자결정),3,긍0·부0·중3,"아리바이오, 아리바이오랩 지배력 강화…30일 유증 납입 후 최대주주로 / 아리바이오, 30일 유증 납...",공시+뉴스
아이텍,KOSDAQ,-15.6,27.8,[투자주의]소수계좌 거래집중 종목,0,,,공시
지슨,KOSDAQ,-13.2,74.8,,0,,,없음
아리바이오홀딩스,KOSDAQ,-12.6,63.4,,0,,,없음
시선AI,KOSDAQ,-9.6,24.7,,0,,,없음
큐리언트,KOSDAQ,-9.5,141.2,[발행조건확정]증권신고서(지분증권) / [기재정정]투자설명서 / 공매도 과열종목 지정(공매도 거래 금...,0,,,공시
크라우드웍스,KOSDAQ,-9.0,14.8,,0,,,없음


In [9]:
table(groups["거래대금 상위"])

,market,change_pct,거래대금(억),공시,뉴스수,긍부정,대표 뉴스,원인 근거
name,,,,,,,,
SK하이닉스,KOSPI,6.7,39018.1,,27,긍2·부6·중19,"SK하이닉스, 협력사 R&D에 1000억원 지원…실패해도 보상 / SK하이닉스, HBM·HBF 경력채...",뉴스
삼성전자,KOSPI,6.7,30006.4,,25,긍1·부5·중19,"삼성전자, 반도체·AI·로봇 ‘신성장동력 인재’ 대거 임원 발탁 / [인사] 삼성전자 외",뉴스
삼성전기,KOSPI,10.3,6675.4,,24,긍3·부0·중21,"삼성 전자계열 42명 승진… AI 수혜 삼성전기 ‘2배’ / 삼성전기, 정기임원 인사… MLCC·기판...",뉴스
SK스퀘어,KOSPI,5.2,4165.5,,0,,,없음
원익홀딩스,KOSDAQ,-2.2,3446.4,,0,,,없음
삼성SDI,KOSPI,13.6,3193.1,임원ㆍ주요주주특정증권등소유상황보고서,26,긍2·부0·중24,[인사] 삼성전자 / 삼성SDI / 삼성디스플레이 등 / 삼성SDI 2027년 임원인사 단행…정태영 ...,공시+뉴스
로보티즈,KOSDAQ,19.5,2090.4,,0,,,없음
주성엔지니어링,KOSDAQ,5.7,2026.3,기업설명회(IR)개최,0,,,공시


## 5. 원인 근거 요약
공시만으로 설명되는 종목, 뉴스로만 설명되는 종목(= CHECK 원본에서는 '이유 없음'으로 남았을 종목)을 셉니다.

In [10]:
summary = pd.concat({k: table(g)["원인 근거"].value_counts() for k, g in groups.items()}, axis=1).fillna(0).astype(int)
summary

,상승 상위,하락 상위,거래대금 상위
원인 근거,,,
공시+뉴스,3,1,1
없음,3,5,3
뉴스,2,0,3
공시,0,2,1


## 결과 읽는 법과 한계
- **공시가 없고 뉴스만 있는 급등**은 테마·풍문·업황 기사로 움직인 경우가 많습니다. 대표 뉴스 제목으로 재료를 확인하세요.
- **긍부정은 기사 단위 모델 값**입니다(`polarity`, 없으면 `customized_ml`의 값). 주가 방향과 반대인 경우도 흔합니다.
- 뉴스 종목 태그는 동명 종목 오탐이 있습니다. 제목에 종목명이 들어간 기사를 우선 보여주지만, 뉴스 수에는 오탐이 섞일 수 있습니다.
- **뉴스가 0건인 급등주도 있습니다.** 상한가·대규모 거래에도 DeepSearch에 수집된 기사가 없거나 종목 태그가 빠진 경우입니다
  (2026-11-24 로보티즈 +19.5%·거래대금 2,090억인데 태그 기사 0건). 이때는 증권사 리포트·커뮤니티 등 다른 출처가 필요합니다.
- 등락률·거래대금은 KRX 기준(NXT 미포함), 가격은 수정주가입니다.